In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


UNSTRUCTURED_API_KEY = os.getenv("UNSTRUCTURED_API_KEY")

In [2]:
file_path = "../input/Gated Attention.pdf"


In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path=file_path,
    strategy="hi_res",
    partition_via_api=True,
    # coordinates = True
    
    # mode="elements",
    # chunking_strategy="by_title",
    # max_characters=800,
    # combine_text_under_n_chars=200,
    # infer_table_structure=True,
    # include_metadata=True
)




In [17]:
docs = []
for doc in loader.lazy_load():
    docs.append(doc)

INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"


NameError: name 'Document' is not defined

In [19]:
docs[0:100]

[Document(metadata={'source': '../input/Gated Attention.pdf', 'coordinates': {'points': [[477.924, 35.21985319999999], [477.924, 45.18245319999994], [524.4094916, 45.18245319999994], [524.4094916, 35.21985319999999]], 'system': 'PixelSpace', 'layout_width': 595.276, 'layout_height': 841.89}, 'page_number': 1, 'languages': ['eng'], 'filename': 'Gated Attention.pdf', 'filetype': 'application/pdf', 'category': 'Header', 'element_id': '293d6998f5ce2fbaeb7fef1d44c41aea'}, page_content='2025-06-11'),
 Document(metadata={'source': '../input/Gated Attention.pdf', 'coordinates': {'points': [[73.55, 70.7008892], [73.55, 100.98808919999999], [523.5185629999997, 100.98808919999999], [523.5185629999997, 70.7008892]], 'system': 'PixelSpace', 'layout_width': 595.276, 'layout_height': 841.89}, 'page_number': 1, 'languages': ['eng'], 'filename': 'Gated Attention.pdf', 'filetype': 'application/pdf', 'parent_id': '293d6998f5ce2fbaeb7fef1d44c41aea', 'category': 'Title', 'element_id': '78c3cc8cf7dcf014c594

In [20]:
print(f"Loaded {len(docs)} documents")
print(f"Sample metadata: {docs[1].metadata if docs else 'No docs'}")


Loaded 208 documents
Sample metadata: {'source': '../input/Gated Attention.pdf', 'is_extracted': 'true', 'coordinates': {'points': [[1326.0623779296875, 94.84130096435547], [1326.0623779296875, 116.85912322998047], [1454.18310546875, 116.85912322998047], [1454.18310546875, 94.84130096435547]], 'system': 'PixelSpace', 'layout_width': 1654, 'layout_height': 2339}, 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'filename': 'Gated Attention.pdf', 'category': 'Header', 'element_id': 'e0f8aad552bfd390c4496e1d7d2a6d14'}


## Import Qdrant Store Module

In [27]:
import sys
sys.path.append('..')  # Add parent directory to path

from qdrant_vectorstore import QdrantStore, create_store_from_documents

In [28]:
# Reload the module to get the latest changes
import importlib
import qdrant_vectorstore
importlib.reload(qdrant_vectorstore)
from qdrant_vectorstore import QdrantStore, create_store_from_documents

In [23]:
# Create store and add documents
store = create_store_from_documents(
    documents=docs,
    collection_name="research_papers_main",
    embedding_model="sentence-transformers/all-MiniLM-L6-v2"
)

# Get collection info
info = store.get_collection_info()
# print(f"Collection Info:")
# print(f"  Points: {info['points_count']}")
# print(f"  Status: {info['status']}")

INFO: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"


Using existing collection: research_papers_main


INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main "HTTP/1.1 200 OK"


Adding 208 documents to Qdrant...


INFO: HTTP Request: PUT https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points?wait=true "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points?wait=true "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points?wait=true "HTTP/1.1 200 OK"


Successfully added 208 documents


INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main "HTTP/1.1 200 OK"


## Load Existing Collection from Qdrant Cloud

This demonstrates how to reconnect to your data later without re-uploading.

In [29]:
# Load existing collection from cloud
print("Loading existing collection...")
store_existing = QdrantStore.from_existing(
    collection_name="research_papers_main",
    embedding_model="sentence-transformers/all-MiniLM-L6-v2"
)

info = store_existing.get_collection_info()
print(f"✓ Loaded collection from Qdrant Cloud")
print(f"  Points: {info['points_count']}")

/home/aman/storage/Python/Projects/Research Paper Assistant/playground/../qdrant_vectorstore.py:83: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
INFO: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading existing collection...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"


Using existing collection: research_papers_main


INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main "HTTP/1.1 200 OK"


✓ Loaded collection from Qdrant Cloud
  Points: 208


In [30]:
store_existing

In [31]:
# First, let's check the collection info and see what metadata is available
info = store_existing.get_collection_info()
print("Collection info:")
print(info)

INFO: HTTP Request: GET https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main "HTTP/1.1 200 OK"


Collection info:
{'name': 'research_papers_main', 'vectors_count': 208, 'points_count': 208, 'status': <CollectionStatus.GREEN: 'green'>}


In [32]:

from langchain_core.documents import Document

# Get all points from the collection
all_points = []
offset = None

while True:
    result = store_existing.client.scroll(
        collection_name=store_existing.collection_name,
        limit=100,
        offset=offset,
        with_payload=True,
        with_vectors=False
    )
    
    points, next_offset = result
    all_points.extend(points)
    
    if next_offset is None:
        break
    offset = next_offset



INFO: HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points/scroll "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points/scroll "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://b9c1abff-fc0c-4fba-8838-f0890f5b091f.us-east4-0.gcp.cloud.qdrant.io:6333/collections/research_papers_main/points/scroll "HTTP/1.1 200 OK"


In [33]:
# Filter by page_number = 1
page_1_points = [p for p in all_points if p.payload.get('metadata', {}).get('page_number') == 1]

print(f"Total points in collection: {len(all_points)}")
print(f"Points with page_number = 1: {len(page_1_points)}")
print("\n" + "="*80 + "\n")

# Display the data
for i, point in enumerate(page_1_points, 1):
    print(f"Document {i}:")
    print(f"  Content: {point.payload.get('page_content', '')[:150]}...")
    print(f"  Metadata: {point.payload.get('metadata', {})}")
    print("-" * 80)

Total points in collection: 208
Points with page_number = 1: 13


Document 1:
  Content: 2025-06-11...
  Metadata: {'source': '../input/Gated Attention.pdf', 'is_extracted': 'true', 'coordinates': {'points': [[1326.0623779296875, 94.84130096435548], [1326.0623779296875, 116.85912322998048], [1454.18310546875, 116.85912322998048], [1454.18310546875, 94.84130096435548]], 'system': 'PixelSpace', 'layout_width': 1654, 'layout_height': 2339}, 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'filename': 'Gated Attention.pdf', 'category': 'Header', 'element_id': 'e0f8aad552bfd390c4496e1d7d2a6d14'}
--------------------------------------------------------------------------------
Document 2:
  Content: Zihan Qiu*!, Zekun Wang*!, Bo Zheng*!, Zeyu Huang*?, Kaiyue Wen?, Songlin Rui Men, Le Yu!, Fei Hu, Suozhi Hu, Dayiheng Liu™!, Jingren Zhou!, Junyang L...
  Metadata: {'source': '../input/Gated Attention.pdf', 'is_extracted': 'true', 'coordinates': {'points': [[341.3212890625,

## Extract Different Content Types by Category

In [34]:
# 1. Extract all IMAGES
print("=" * 80)
print("IMAGES")
print("=" * 80)

images = [p for p in all_points if p.payload.get('metadata', {}).get('category') == 'Image']

print(f"\nTotal Images Found: {len(images)}\n")

for i, point in enumerate(images, 1):
    metadata = point.payload.get('metadata', {})
    print(f"Image {i}:")
    print(f"  Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"  Source: {metadata.get('source', 'N/A')}")
    print(f"  Element ID: {metadata.get('element_id', 'N/A')}")
    print(f"  Content Preview: {point.payload.get('page_content', '')[:100]}")
    print("-" * 80)

IMAGES

Total Images Found: 10

Image 1:
  Page Number: 7
  Source: ../input/Gated Attention.pdf
  Element ID: 618d8f148aae5341f7338ddd265553ca
  Content Preview: SDPA_Elementwise HS_Gate Scores Distribution 0.16 T 0.14 = Distribution --- Mean =0.271 0.12 §0.10 g
--------------------------------------------------------------------------------
Image 2:
  Page Number: 14
  Source: ../input/Gated Attention.pdf
  Element ID: 6d69c4dd2b617479e949625be152cde6
  Content Preview: Hidden State Mean - hidden state mean before gate, avg 0.71 hidden state mean after gate, avg 0.05 —
--------------------------------------------------------------------------------
Image 3:
  Page Number: 14
  Source: ../input/Gated Attention.pdf
  Element ID: 6441d3e71d5fd63b9b4ac0aad0b33fee
  Content Preview: Sparsity Ratio of SDPA Output, Threshold le-2 Sparsity Ratio of SDPA Output, Threshold le-3 1.0 -~ s
--------------------------------------------------------------------------------
Image 4:
  Page Number: 1
 

In [30]:
# 2. Extract all FORMULAS
print("=" * 80)
print("FORMULAS")
print("=" * 80)

formulas = [p for p in all_points if p.payload.get('metadata', {}).get('category') == 'Formula']

print(f"\nTotal Formulas Found: {len(formulas)}\n")

for i, point in enumerate(formulas, 1):
    metadata = point.payload.get('metadata', {})
    print(f"Formula {i}:")
    print(f"  Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"  Source: {metadata.get('source', 'N/A')}")
    print(f"  Element ID: {metadata.get('element_id', 'N/A')}")
    print(f"  Content: {point.payload.get('page_content', '')}")
    print("-" * 80)

FORMULAS

Total Formulas Found: 8

Formula 1:
  Page Number: 3
  Source: ../input/Gated Attention.pdf
  Element ID: 3e67a56dd7a7449ddaf432621b27d1db
  Content: Y ′ = g(Y ,X,Wθ,σ) = Y ⊙ σ(XWθ), (5)
--------------------------------------------------------------------------------
Formula 2:
  Page Number: 2
  Source: ../input/Gated Attention.pdf
  Element ID: effc309511043e90b2381f64fc5988e1
  Content: Q = XWQ, K = XWK, V = XWV . (1)
--------------------------------------------------------------------------------
Formula 3:
  Page Number: 6
  Source: ../input/Gated Attention.pdf
  Element ID: db2d8019f1b789433c0d4a475cd7b008
  Content: k S, 4, (8) = Non-Linearity-Map - W,
--------------------------------------------------------------------------------
Formula 4:
  Page Number: 6
  Source: ../input/Gated Attention.pdf
  Element ID: bdcaf43a25616a23266cc5e4a4095f42
  Content: ) - Non-Linearity-Map (X W Wk,
--------------------------------------------------------------------------------
Form

In [35]:
# 3. Extract all TABLES
print("=" * 80)
print("TABLES")
print("=" * 80)

tables = [p for p in all_points if p.payload.get('metadata', {}).get('category') == 'Table']

print(f"\nTotal Tables Found: {len(tables)}\n")

for i, point in enumerate(tables, 1):
    metadata = point.payload.get('metadata', {})
    print(f"Table {i}:")
    print(f"  Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"  Source: {metadata.get('source', 'N/A')}")
    print(f"  Element ID: {metadata.get('element_id', 'N/A')}")
    print(f"  Content Preview: {point.payload.get('page_content', '')[:200]}")
    print("-" * 80)

TABLES

Total Tables Found: 6

Table 1:
  Page Number: 4
  Source: ../input/Gated Attention.pdf
  Element ID: 6bfc41074f8e70d18e766a54eb9225e5
  Content Preview: Method Act Func Score Shape Added Param Avg PPL Hellaswag MMLU GSM8k C-eval Reference Baselines (Baseline uses q = 32,k = 4. All methods use dk = 128.) (1) Baseline - - 0 6.026 73.07 58.79 52.92 (2) k
--------------------------------------------------------------------------------
Table 2:
  Page Number: 5
  Source: ../input/Gated Attention.pdf
  Element ID: 68ae31338ccba8502506d6cd48de4f15
  Content Preview: Method Max LR Avg PPL HumanEval MMLU GSM8k Hellaswag C-eval CMMLU 28 Layer, 1.7B Parameters, 400B Tokens, Batch Size=1024 (1) Baseline (2) SDPA Elementwise 4.0 × 10−3 4.0 × 10−3 7.499 7.404 28.66 29.2
--------------------------------------------------------------------------------
Table 3:
  Page Number: 8
  Source: ../input/Gated Attention.pdf
  Element ID: f5feb88a18d95ba15f6072063901c220
  Content Preview: Method Basel

In [32]:
# 4. Extract all TITLES
print("=" * 80)
print("TITLES")
print("=" * 80)

titles = [p for p in all_points if p.payload.get('metadata', {}).get('category') == 'Title']

print(f"\nTotal Titles Found: {len(titles)}\n")

for i, point in enumerate(titles, 1):
    metadata = point.payload.get('metadata', {})
    print(f"Title {i}:")
    print(f"  Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"  Source: {metadata.get('source', 'N/A')}")
    print(f"  Element ID: {metadata.get('element_id', 'N/A')}")
    print(f"  Content: {point.payload.get('page_content', '')}")
    print("-" * 80)

TITLES

Total Titles Found: 27

Title 1:
  Page Number: 14
  Source: ../input/Gated Attention.pdf
  Element ID: 182038af5456c3ccedcb19a3f4aa2e11
  Content: A.4 More Layerwise Gating Scores
--------------------------------------------------------------------------------
Title 2:
  Page Number: 4
  Source: ../input/Gated Attention.pdf
  Element ID: 6d05d6aa4cf8fbf18cde0243865622e6
  Content: 3.2 Main Results
--------------------------------------------------------------------------------
Title 3:
  Page Number: 13
  Source: ../input/Gated Attention.pdf
  Element ID: e11933b84ed4bd634be26842db72ec86
  Content: A.1 Switch Head Baselines
--------------------------------------------------------------------------------
Title 4:
  Page Number: 13
  Source: ../input/Gated Attention.pdf
  Element ID: c54149cbb1d67644283ad7dbb49cbcaa
  Content: A.2 More Discussion on Sparse Gating Score
--------------------------------------------------------------------------------
Title 5:
  Page Number: 1
  So

In [33]:
# 5. Summary of all categories
print("=" * 80)
print("SUMMARY: ALL CATEGORIES")
print("=" * 80)

# Get all unique categories
categories = {}
for point in all_points:
    category = point.payload.get('metadata', {}).get('category', 'Unknown')
    categories[category] = categories.get(category, 0) + 1

print(f"\nTotal Documents: {len(all_points)}\n")
print("Category Distribution:")
for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {count}")
print("=" * 80)

SUMMARY: ALL CATEGORIES

Total Documents: 208

Category Distribution:
  NarrativeText: 63
  ListItem: 58
  Title: 27
  Footer: 19
  FigureCaption: 12
  Image: 10
  Formula: 8
  Table: 6
  UncategorizedText: 3
  Header: 2


In [35]:
# 6. Filter by BOTH category and page number
# Example: Get all titles from page 1

target_category = "Title"  # Change to: "Image", "Formula", "Table", "CompositeElement", etc.
target_page = 1  # Change to desired page number

filtered_results = [
    p for p in all_points 
    if p.payload.get('metadata', {}).get('category') == target_category 
    
]

print("=" * 80)
print(f"FILTERED RESULTS: {target_category} from Page {target_page}")
print("=" * 80)
print(f"\nFound {len(filtered_results)} items\n")

for i, point in enumerate(filtered_results, 1):
    metadata = point.payload.get('metadata', {})
    print(f"{target_category} {i}:")
    print(f"  Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"  Element ID: {metadata.get('element_id', 'N/A')}")
    print(f"  Content: {point.payload.get('page_content', '')}")
    print("-" * 80)

FILTERED RESULTS: Title from Page 1

Found 27 items

Title 1:
  Page Number: 14
  Element ID: 182038af5456c3ccedcb19a3f4aa2e11
  Content: A.4 More Layerwise Gating Scores
--------------------------------------------------------------------------------
Title 2:
  Page Number: 4
  Element ID: 6d05d6aa4cf8fbf18cde0243865622e6
  Content: 3.2 Main Results
--------------------------------------------------------------------------------
Title 3:
  Page Number: 13
  Element ID: e11933b84ed4bd634be26842db72ec86
  Content: A.1 Switch Head Baselines
--------------------------------------------------------------------------------
Title 4:
  Page Number: 13
  Element ID: c54149cbb1d67644283ad7dbb49cbcaa
  Content: A.2 More Discussion on Sparse Gating Score
--------------------------------------------------------------------------------
Title 5:
  Page Number: 1
  Element ID: 319ddf828c95298415d9f0e04839bfbc
  Content: 1 Introduction
------------------------------------------------------------------

## Search Data by Element ID

Search for specific documents using their element IDs.

In [19]:
def search_by_element_id(element_id, all_points):
    """
    Search for a document by its element_id.
    
    Args:
        element_id: The element_id to search for
        all_points: List of all points from Qdrant collection
        
    Returns:
        The matching point or None if not found
    """
    for point in all_points:
        if point.payload.get('metadata', {}).get('element_id') == element_id:
            return point
    return None


def search_by_element_ids(element_ids, all_points):
    """
    Search for multiple documents by their element_ids.
    
    Args:
        element_ids: List of element_ids to search for
        all_points: List of all points from Qdrant collection
        
    Returns:
        List of matching points
    """
    matching_points = []
    element_id_set = set(element_ids)
    
    for point in all_points:
        point_element_id = point.payload.get('metadata', {}).get('element_id')
        if point_element_id in element_id_set:
            matching_points.append(point)
    
    return matching_points


# Example: Search for a single element_id
example_element_id = "dd11127072238810faea64f51a979b80"  # Replace with your element_id




result = search_by_element_id(example_element_id, all_points)

if result:
    print(f"✓ Found document with element_id: {example_element_id}")
    print(f"\nCategory: {result.payload.get('metadata', {}).get('category')}")
    print(f"Page Number: {result.payload.get('metadata', {}).get('page_number')}")
    print(f"Content: {result.payload.get('page_content', '')}...")
    
else:
    print(f"✗ No document found with element_id: {example_element_id}")

✓ Found document with element_id: dd11127072238810faea64f51a979b80

Category: NarrativeText
Page Number: 1
Content: Gating mechanisms have been widely utilized, from early models like LSTMs (Hochreiter & Schmidhuber, 1997) and Highway Networks (Srivastava et al., 2015) to recent state space models (Gu & Dao, 2023), linear attention (Hua et al., 2022), and also softmax attention (Lin et al., 2025). Yet, existing literature rarely examines the specific effects of gating. In this work, we conduct comprehensive experiments to systematically investigate gating-augmented softmax attention variants. Specifically, we perform a comprehen- sive comparison over 30 variants of 15B Mixture-of-Experts (MoE) models and 1.7B dense models trained on a 3.5 trillion token dataset. Our central finding is that a simple modification—applying a head-specific sigmoid gate after the Scaled Dot-Product Attention (SDPA)—consistently improves performance. This modification also enhances training sta- bility, tole

In [10]:
# Example: Search for multiple element_ids at once
# These are the abstract text block IDs from the metadata extraction
element_ids_to_search = [
    'ffdc80f0680c881886931d2311a1777a',
    'dd11127072238810faea64f51a979b80'

]

results = search_by_element_ids(element_ids_to_search, all_points)

print(f"✓ Found {len(results)} documents\n")
print("=" * 80)

for i, point in enumerate(results, 1):
    metadata = point.payload.get('metadata', {})
    print(f"\nDocument {i}:")
    print(f"  Element ID: {metadata.get('element_id')}")
    print(f"  Category: {metadata.get('category')}")
    print(f"  Page Number: {metadata.get('page_number')}")
    print(f"  Content: {point.payload.get('page_content', '')}...")
    
    print("-" * 80)

✓ Found 2 documents


Document 1:
  Element ID: ffdc80f0680c881886931d2311a1777a
  Category: NarrativeText
  Page Number: 1
  Content: Zihan Qiu*!, Zekun Wang*!, Bo Zheng*!, Zeyu Huang*?, Kaiyue Wen?, Songlin Rui Men, Le Yu!, Fei Hu, Suozhi Hu, Dayiheng Liu™!, Jingren Zhou!, Junyang Lin™! 1Qwen Team, Alibaba Group Uv’ of Edinburgh 3Stanford University AMIT shu University...
--------------------------------------------------------------------------------

Document 2:
  Element ID: dd11127072238810faea64f51a979b80
  Category: NarrativeText
  Page Number: 1
  Content: Gating mechanisms have been widely utilized, from early models like LSTMs (Hochreiter & Schmidhuber, 1997) and Highway Networks (Srivastava et al., 2015) to recent state space models (Gu & Dao, 2023), linear attention (Hua et al., 2022), and also softmax attention (Lin et al., 2025). Yet, existing literature rarely examines the specific effects of gating. In this work, we conduct comprehensive experiments to systematically i

In [ ]:
# Advanced: Get full details including coordinates for element_ids
def get_full_details_by_element_ids(element_ids, all_points):
    """
    Get full details including coordinates for documents by element_ids.
    
    Args:
        element_ids: List of element_ids to search for
        all_points: List of all points from Qdrant collection
        
    Returns:
        List of dictionaries with full document details
    """
    results = []
    element_id_set = set(element_ids)
    
    for point in all_points:
        metadata = point.payload.get('metadata', {})
        point_element_id = metadata.get('element_id')
        
        if point_element_id in element_id_set:
            results.append({
                'element_id': point_element_id,
                'category': metadata.get('category'),
                'page_number': metadata.get('page_number'),
                'content': point.payload.get('page_content', ''),
                'coordinates': metadata.get('coordinates', {}),
                'parent_id': metadata.get('parent_id'),
                'source': metadata.get('source'),
                'full_metadata': metadata
            })
    
    return results


# Example usage with abstract element IDs
abstract_element_ids = [
    'ffdc80f0680c881886931d2311a1777a',
    'dd11127072238810faea64f51a979b80'
]

details = get_full_details_by_element_ids(abstract_element_ids, all_points)

print(f"✓ Retrieved {len(details)} documents with full details\n")
print("=" * 80)

for i, doc in enumerate(details, 1):
    print(f"\nDocument {i}:")
    print(f"  Element ID: {doc['element_id']}")
    print(f"  Category: {doc['category']}")
    print(f"  Page: {doc['page_number']}")
    print(f"  Coordinates: {doc['coordinates']}")
    print(f"  Content: {doc['content'][:200]}...")
    print("-" * 80)